In [1]:
%pip install requests pandas lxml openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests 
import pandas as pd
from io import StringIO

In [3]:
url = "https://stockanalysis.com/quote/egx/EFID/financials/cash-flow-statement/?p=quarterly"
response = requests.get(url)

response.status_code

200

In [4]:
companiesEGX = [
    "COMI", "SWDY", "ETEL", "TMGH", "EGAL", "MFPC", "QNBE", "ABUK",
    "HDBK", "EAST", "ALCN", "ORAS", "EFIH", "EMFD", "ADIB", "FWRY",
    "SCTS", "ORHD", "EFID", "CANA", "OCDI", "GPPL", "VLMR", "VLMRA",
    "PHDC", "JUFO", "HRHO", "GBCO", "HELI", "CIEB", "BTFH", "FERC",
    "FAIT", "FAITA", "RAYA", "EXPA", "ARCC", "IRON", "EGCH", "SCEM",
    "CCAP", "BIOC", "CLHO", "VALU", "MCQE", "MBSC", "CIRA", "EFIC",
    "PHAR", "TAQA", "SKPC", "MTIE", "POUL", "ORWE", "EGTS", "UBEE",
    "MASR", "AMOC", "EGSA", "SAUD", "NIPH", "MOIL", "ATQA", "KORA",
    "AMES", "MHOT", "EGBE", "TALM", "ISPH", "CICH", "RMDA", "CSAG",
    "OIH", "BINV", "IFAP", "MOIN", "ZMID", "AMIA", "MPCI", "MIPH",
    "OLFI", "MPRC", "SUGR", "ISMQ", "PRDC", "BONY", "EGAS", "AXPH",
    "PHTV", "CPCI", "DOMT", "GOUR", "ELEC", "NINH", "SPIN", "ACAP",
    "NAPR", "ENGC", "SPHT", "ARAB", "OCPH", "SVCE", "CNFN", "MICH",
    "KABO", "AFMC", "GSSC", "WCDF", "DSCW", "OFH", "AMER", "MFSC",
    "ACGC", "SAIB", "GDWA", "UNIT", "KZPC", "SDTI", "CRST", "UEFM",
    "AJWA", "ADCI", "GTWL", "ELKA", "ACTF", "ASCM", "ELSH", "DAPH",
    "INFI", "ISMA", "ATLC", "ALRA", "CFGH", "ZEOT", "LCSW", "ICFC",
    "GPIM", "NAHO", "MPCO", "ACAMD", "ETRS", "EDFM", "SMFR", "NARE",
    "CEFM", "MILS", "PHGC", "GGRN", "IDRE", "AALR", "EHDR", "EALR",
    "ADPC", "RACC", "NCCW", "WKOL", "GGCC", "ICID", "MOSC", "MAAL",
    "SNFC", "CERA", "ECAP", "ODIN", "ASPI", "KRDI", "DTPP", "SCFM",
    "MENA", "PRCL", "CAED", "SIPC", "DEIN", "MBEG", "LUTS", "MEPA",
    "OBRI", "NHPS", "IEEC", "AIHC", "UEGC", "SEIG", "SEIGA", "NDRL",
    "COSG", "ALUM", "AIDC", "EBSC", "AMII", "RREI", "GTEX", "POCO",
    "RTVC", "AFDI", "MCRO", "TANM", "PRMH", "UNIP", "EASB", "TWSA",
    "APSW", "MOED", "RUBX", "ICLE", "MEGM", "TYCN", "ROTO", "SPMD",
    "KWIN", "RAKT", "EEII", "AREH", "GRCA", "CCRS", "EPCO", "GIHD",
    "FCMD", "TRTO", "ELWA", "ELNA", "DGTZ", "DCCC", "MMAT", "NEDA",
    "EPPK", "GMCI", "EOSB", "CPME", "COPR"
]


statements = {
    "income-statement" : 1 ,
    "balance-sheet" : 3 , 
    "cash-flow-statement" : 3
}

In [5]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
}

all_data = []

for company in companiesEGX:
    for statement , n_tables in statements.items() :
        try:
           url = f"https://stockanalysis.com/quote/egx/{company}/financials/{statement}/?p=quarterly"
           html_tables = requests.get(url , headers= headers).text
           tables = pd.read_html(StringIO(html_tables))[:n_tables]

           for df in tables:
            df.columns = df.columns.get_level_values(-1)
            df = df.rename(columns = {df.columns[0] : "Financial Term"})
            df = df.melt(id_vars = "Financial Term" , var_name = "End Of Quarter" , value_name = "Amount")
            df["Company"] = company
            df["Statement Type"] = statement
            all_data.append(df)

        except Exception as e:
           continue

In [6]:
data = pd.concat(all_data , ignore_index= True)
data["End Of Quarter"] = pd.to_datetime(
    data["End Of Quarter"].str.extract(r"(\w{3} \d{1,2}, \d{4})")[0] , format="%b %d, %Y"
)

data["Amount"] = pd.to_numeric(
    data["Amount"].str.replace("," , "") , errors= "coerce"
)

data = data.dropna(subset= ["End Of Quarter" , "Amount"])
data

,Financial Term,End Of Quarter,Amount,Company,Statement Type
0,Interest Income on Loans,2026-06-30,57262.00,COMI,income-statement
1,Interest Income on Investments,2026-06-30,161.26,COMI,income-statement
2,Total Interest Income,2026-06-30,57424.00,COMI,income-statement
3,Interest Paid on Deposits,2026-06-30,26136.00,COMI,income-statement
4,Net Interest Income,2026-06-30,31288.00,COMI,income-statement
...,...,...,...,...,...
182537,Net Income to Common,2021-06-30,13.80,GSSC,income-statement
182539,Shares Outstanding (Basic),2021-06-30,18.00,GSSC,income-statement
182540,Shares Outstanding (Diluted),2021-06-30,18.00,GSSC,income-statement
182542,EPS (Basic),2021-06-30,0.79,GSSC,income-statement


In [7]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = "localhost"
database = "EGX"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string)
)

In [9]:
data.to_sql(
    "FactFinancial",
    con=engine,
    if_exists="replace",
    index=False
)

319